In [5]:
!git clone https://github.com/MuhammadBramansyah/Interview_section.git

Cloning into 'Interview_section'...
remote: Enumerating objects: 3971, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3971 (delta 0), reused 0 (delta 0), pack-reused 3969 (from 3)
Receiving objects: 100% (3971/3971), 93.76 MiB | 33.87 MiB/s, done.
Resolving deltas: 100% (1930/1930), done.


In [14]:
!nvidia-smi

Fri Dec 19 06:58:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%cd Interview_section

[Errno 2] No such file or directory: 'Interview_section'
/content


In [4]:
!git clone https://github.com/MuhammadBramansyah/Interview_section.git

Cloning into 'Interview_section'...
remote: Enumerating objects: 3971, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3971 (delta 0), reused 0 (delta 0), pack-reused 3969 (from 3)
Receiving objects: 100% (3971/3971), 93.76 MiB | 24.26 MiB/s, done.
Resolving deltas: 100% (1930/1930), done.


In [6]:
%cd /content

/content


In [12]:
!ls

dataset_people


In [11]:
%cd /content/Interview_section

/content/Interview_section


In [15]:
## dataloader
import os
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import xml.etree.ElementTree as ET


class PascalVOCDataset(Dataset):
    def __init__(self, root_dir):
        self.img_dir = os.path.join(root_dir, "images")
        self.ann_dir = os.path.join(root_dir, "labels")

        self.images = [
            f for f in os.listdir(self.img_dir)
            if f.lower().endswith((".jpg", ".png", ".jpeg"))
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]

        img_path = os.path.join(self.img_dir, img_name)
        ann_path = os.path.join(
            self.ann_dir,
            os.path.splitext(img_name)[0] + ".xml"
        )

        image = Image.open(img_path).convert("RGB")
        width, height = image.size

        boxes = []
        labels = []

        if os.path.exists(ann_path):
            tree = ET.parse(ann_path)
            root = tree.getroot()

            for obj in root.findall("object"):
                if obj.find("name").text != "person":
                    continue

                box = obj.find("bndbox")

                xmin = max(0, int(box.find("xmin").text))
                ymin = max(0, int(box.find("ymin").text))
                xmax = min(width, int(box.find("xmax").text))
                ymax = min(height, int(box.find("ymax").text))

                if xmax <= xmin or ymax <= ymin:
                    continue

                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(1)  # person = 1

        # ===== HANDLE EMPTY ANNOTATION =====
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (
                (boxes[:, 2] - boxes[:, 0]) *
                (boxes[:, 3] - boxes[:, 1])
                if len(boxes) > 0 else torch.tensor([])
            ),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
        }

        image = torchvision.transforms.ToTensor()(image)

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


def create_dataloader(path, batch_size, shuffle=True):
    dataset = PascalVOCDataset(path)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_fn,
        # num_workers=2,
        pin_memory=True,
    )


In [16]:
## model
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(
        in_features, num_classes
    )
    return model


In [ ]:
## train
import torch
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchvision.ops import box_iou

# from dataset_loader import create_dataloader
# from model import get_model

# ================= CONFIG =================
DATASET_DIR = "/content/Interview_section/dataset_people"   # 🔥 SESUAI DATASET ANDA
NUM_CLASSES = 2
BATCH_SIZE = 2
EPOCHS = 50
LR = 0.005
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONF_THRESH = 0.05
IOU_THRESH = 0.5

# ===== EARLY STOPPING CONFIG =====
PATIENCE = 25
MIN_DELTA = 1e-4
BEST_MODEL_OUT = "best_faster_rcnn_people.pt"
LAST_MODEL_OUT = "last_faster_rcnn_people.pt"

# ===== CHART DIR =====
CHART_DIR = "chart"
os.makedirs(CHART_DIR, exist_ok=True)

# ================= LOAD DATA =================
train_loader = create_dataloader(
    os.path.join(DATASET_DIR, "train"),
    BATCH_SIZE,
    shuffle=True
)

val_loader = create_dataloader(
    os.path.join(DATASET_DIR, "valid"),  # 🔥 valid (bukan val)
    BATCH_SIZE,
    shuffle=False
)

# ================= MODEL =================
model = get_model(NUM_CLASSES)
model.to(DEVICE)

# ================= MODEL SUMMARY =================
def model_summary(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("\n========== MODEL ARCHITECTURE ==========")
    print(model)
    print("=======================================")
    print("\n========== MODEL SUMMARY ===============")
    print(f"Total parameters     : {total:,}")
    print(f"Trainable parameters : {trainable:,}")
    print("=======================================\n")

model_summary(model)

# ================= EVALUATION FUNCTION =================
def evaluate_map(model, dataloader):
    model.eval()
    total_tp, total_fp, total_fn = 0, 0, 0

    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Evaluating", leave=False):
            images = [img.to(DEVICE) for img in images]
            outputs = model(images)

            for output, target in zip(outputs, targets):
                gt_boxes = target["boxes"].to(DEVICE)
                pred_boxes = output["boxes"].to(DEVICE)
                scores = output["scores"].to(DEVICE)

                keep = scores >= CONF_THRESH
                pred_boxes = pred_boxes[keep]

                if len(pred_boxes) == 0:
                    total_fn += len(gt_boxes)
                    continue

                if len(gt_boxes) == 0:
                    total_fp += len(pred_boxes)
                    continue

                ious = box_iou(pred_boxes, gt_boxes)

                matched_gt = set()
                matched_pred = set()

                for _ in range(min(len(pred_boxes), len(gt_boxes))):
                    max_iou, _ = ious.max(dim=1)
                    best_pred = torch.argmax(max_iou).item()
                    best_gt = torch.argmax(ious[best_pred]).item()

                    if ious[best_pred, best_gt] < IOU_THRESH:
                        break

                    matched_pred.add(best_pred)
                    matched_gt.add(best_gt)

                    ious[best_pred, :] = -1
                    ious[:, best_gt] = -1

                tp = len(matched_gt)
                fp = len(pred_boxes) - tp
                fn = len(gt_boxes) - tp

                total_tp += tp
                total_fp += fp
                total_fn += fn

    precision = total_tp / (total_tp + total_fp + 1e-6)
    recall = total_tp / (total_tp + total_fn + 1e-6)

    return precision, recall  # precision = pseudo mAP@0.5

# ================= OPTIMIZER =================
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    momentum=0.9,
    weight_decay=0.0005,
)

# ================= METRIC STORAGE =================
train_losses = []
val_losses = []
map50_scores = []

best_val_loss = float("inf")
best_epoch = -1

# ================= TRAIN =================
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for images, targets in tqdm(train_loader, desc=f"Train Epoch {epoch+1}"):
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # ===== VALIDATION LOSS =====
    model.train()  # FasterRCNN requirement
    val_loss = 0.0

    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Val Loss", leave=False):
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            val_loss += sum(loss for loss in loss_dict.values()).item()

    # ===== VALIDATION METRIC =====
    map50, recall = evaluate_map(model, val_loader)

    # ===== STORE METRICS =====
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    map50_scores.append(map50)

    print(
        f"\nEpoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"mAP@0.5: {map50:.4f} | "
        f"Recall: {recall:.4f}"
    )

    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        torch.save(model.state_dict(), BEST_MODEL_OUT)
        print(f"✅ Best model saved at epoch {best_epoch}")

# ================= CHART ANALYSIS =================
epochs = range(1, len(train_losses) + 1)

plt.figure()
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss")
plt.legend()
plt.grid()
plt.savefig(os.path.join(CHART_DIR, "loss_curve.png"))
plt.close()

plt.figure()
plt.plot(epochs, map50_scores)
plt.xlabel("Epoch")
plt.ylabel("mAP@0.5")
plt.title("mAP@0.5 per Epoch")
plt.grid()
plt.savefig(os.path.join(CHART_DIR, "map50_curve.png"))
plt.close()

print(f"\n📊 Charts saved in folder: {CHART_DIR}/")

# ================= FINAL EVALUATION =================
model.load_state_dict(torch.load(BEST_MODEL_OUT, map_location=DEVICE))
final_map, final_recall = evaluate_map(model, val_loader)

print("\n========== FINAL MODEL EVALUATION ==========")
print(f"Best Epoch       : {best_epoch}")
print(f"Final mAP@0.5    : {final_map:.4f}")
print(f"Final Recall     : {final_recall:.4f}")
print("============================================")

torch.save(model.state_dict(), LAST_MODEL_OUT)
print(f"\n💾 Last model saved: {LAST_MODEL_OUT}")
print(f"🏆 Best model saved: {BEST_MODEL_OUT}")


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 192MB/s] 



========== MODEL ARCHITECTURE ==========
FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNo

Train Epoch 1: 100%|██████████| 784/784 [05:21<00:00,  2.44it/s]



Epoch 1/50 | Train Loss: 380.6947 | Val Loss: 42.2440 | mAP@0.5: 0.2747 | Recall: 0.9759
✅ Best model saved at epoch 1


Train Epoch 2: 100%|██████████| 784/784 [05:22<00:00,  2.43it/s]


... (40 outputs left out)


Epoch 22/50 | Train Loss: 111.0574 | Val Loss: 41.5334 | mAP@0.5: 0.7181 | Recall: 0.9440


Train Epoch 23: 100%|██████████| 784/784 [05:23<00:00,  2.43it/s]



Epoch 23/50 | Train Loss: 111.1392 | Val Loss: 43.5324 | mAP@0.5: 0.6891 | Recall: 0.9384


Val Loss:  87%|████████▋ | 87/100 [00:15<00:02,  5.52it/s]